### **Mount to the Drive**

In [76]:
from google.colab import drive
drive.mount('/content/drive')

base_dir = '/content/drive/MyDrive/epipolar-geometry-dinov2/epipolar-geometry-dinov2'

%cd $base_dir

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/epipolar-geometry-dinov2/epipolar-geometry-dinov2


In [77]:
# Run the command below if you need
!pip install -q -r requirements.txt

In [78]:
import cv2 as cv
import numpy as np
import pandas as pd
from src.evaluation import symmetric_epipolar_distance, sampson_error
from src.geometry import calculate_fundamental_matrix, calculate_fundamental_matrix_from_pts
from src.feature_extraction import extract_sift_features, knn_matcher, load_dinov2_model, extract_dinov2_features_stereo, match_dinov2_features

In [79]:
left_img_path = f'{base_dir}/data/kitti_sample/sequences/00/image_2/000000.png'
right_img_path = f'{base_dir}/data/kitti_sample/sequences/00/image_3/000000.png'

left_img = cv.imread(left_img_path)
right_img = cv.imread(right_img_path)

# **SIFT**

In [80]:
sift_left_kps, sift_left_des = extract_sift_features(left_img)
sift_right_kps, sift_right_des = extract_sift_features(right_img)

good = knn_matcher(sift_left_des, sift_right_des)

sift_F, sift_pts_left, sift_pts_right = calculate_fundamental_matrix(good, (sift_left_kps, sift_right_kps))

In [81]:
sift_sed_err = symmetric_epipolar_distance(sift_pts_left, sift_pts_right, sift_F)
sift_sampson_err = sampson_error(sift_pts_left, sift_pts_right, sift_F)

# **DINOv2**

In [82]:
processor, model, patch_size = load_dinov2_model()
(resized_img_left, resized_img_right), normalized_patch_features, num_patches_width, num_patches_flat = extract_dinov2_features_stereo((left_img, right_img), processor, model, patch_size)

dino_pts_left, dino_pts_right = match_dinov2_features(normalized_patch_features, patch_size, num_patches_width, num_patches_flat)
dino_F, (dino_inliers1, dino_inliers2) = calculate_fundamental_matrix_from_pts(dino_pts_left, dino_pts_right)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [83]:
dino_sed_err = symmetric_epipolar_distance(dino_inliers1, dino_inliers2, dino_F)
dino_sampson_err = sampson_error(dino_inliers1, dino_inliers2, dino_F)

In [85]:
results = {
    "Method": ["SIFT", "DINOv2"],
    "Inlier Points": [len(sift_pts_left), len(dino_inliers1)],
    "Median Symmetric Dist.": [np.median(sift_sed_err), np.median(dino_sed_err)],
    "Median Sampson Err.": [np.median(sift_sampson_err), np.median(dino_sampson_err)],
    "Mean Sampson Err.": [np.mean(sift_sampson_err), np.mean(dino_sampson_err)],
    "Max Sampson Err.": [np.max(sift_sampson_err), np.max(dino_sampson_err)]
}

df = pd.DataFrame(results)
df.set_index("Method", inplace=True)
display(df)

,Inlier Points,Median Symmetric Dist.,Median Sampson Err.,Mean Sampson Err.,Max Sampson Err.
Method,,,,,
SIFT,925,1.749018e+00,4.434024e-01,7.522841e-01,4.401623e+00
DINOv2,965,2.323716e-20,5.915444e-21,9.867080e-21,6.021459e-20
